# LeetCode 1940 — Longest Common Subsequence Between Sorted Arrays bằng Python

Link đề: https://leetcode.com/problems/longest-common-subsequence-between-sorted-arrays/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Counting

> 🔒 **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: kho lời giải doocs/leetcode). Nếu bạn có tài khoản Premium, nên đối chiếu lại câu chữ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def longestCommonSubsequence(self, arrays: List[List[int]]) -> List[int]:
        ...
```

Lộ trình: **DP LCS cổ điển áp lần lượt cho từng cặp** (đúng nhờ mảng đã sắp xếp) → **giao hai con trỏ lần lượt** `O(Σ len)` → **giao tập hợp rồi sắp xếp** → **đếm tần suất** (giá trị ≤ 100) `O(Σ len + 100)` (lời giải nộp). Mấu chốt: với các mảng **tăng ngặt**, dãy con chung dài nhất chính là **giao** các mảng theo thứ tự tăng.

## 1. Mục tiêu học

1. Ôn **DP dãy con chung dài nhất (LCS)** của hai dãy và thấy vì sao nó tổng quát nhưng chậm.
2. Nhận ra cấu trúc đặc biệt: mảng **tăng ngặt** ⇒ mọi tập con lấy theo thứ tự tăng đều là dãy con ⇒ LCS = **giao các tập**.
3. Giao hai mảng đã sắp xếp bằng **hai con trỏ**; giao nhiều mảng bằng cách **gộp dần**.
4. Dùng **đếm tần suất** khi miền giá trị nhỏ (`≤ 100`).
5. Biết khi nào ý tưởng "giao" **không** còn đúng (mảng không sắp xếp) — và kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Cho mảng các mảng số nguyên `arrays`, mỗi `arrays[i]` **tăng ngặt**. Trả về mảng biểu diễn **dãy con chung dài nhất** của **tất cả** các mảng.

(Dãy con: bỏ một số phần tử, giữ nguyên thứ tự phần tử còn lại.)

**Ví dụ 1:** `[[1,3,4],[1,4,7,9]]` → `[1,4]`.

**Ví dụ 2:** `[[2,3,6,8],[1,2,3,5,6,7,10],[2,3,4,6,9]]` → `[2,3,6]`.

**Ví dụ 3:** `[[1,2,3,4,5],[6,7,8]]` → `[]`.

**Ràng buộc:** `2 ≤ arrays.length ≤ 100`, `1 ≤ arrays[i].length ≤ 100`, `1 ≤ arrays[i][j] ≤ 100`, mỗi `arrays[i]` tăng ngặt.

In [1]:
import random
import time
from collections import Counter
from functools import reduce
from itertools import combinations
from typing import List

examples = [
    ([[1, 3, 4], [1, 4, 7, 9]], [1, 4]),
    ([[2, 3, 6, 8], [1, 2, 3, 5, 6, 7, 10], [2, 3, 4, 6, 9]], [2, 3, 6]),
    ([[1, 2, 3, 4, 5], [6, 7, 8]], []),
]
for arrs, want in examples:
    print(arrs, "->", want)

def is_subsequence(sub, arr):
    it = iter(arr)
    return all(x in it for x in sub)

print(is_subsequence([1, 4], [1, 3, 4]), is_subsequence([4, 1], [1, 3, 4]))

[[1, 3, 4], [1, 4, 7, 9]] -> [1, 4]
[[2, 3, 6, 8], [1, 2, 3, 5, 6, 7, 10], [2, 3, 4, 6, 9]] -> [2, 3, 6]
[[1, 2, 3, 4, 5], [6, 7, 8]] -> []
True False


## 3. Kiến thức nền

### 3.1 Kiểm tra dãy con bằng iterator

`x in it` tiêu thụ iterator tới khi gặp `x` — nên `all(x in it for x in sub)` kiểm `sub` là dãy con của `arr` trong `O(len(arr))`.

In [2]:
it = iter([1, 2, 3, 4])
print(2 in it, 1 in it)        # sau khi tìm thấy 2, iterator đã đi qua 1

True False


### 3.2 DP LCS của hai dãy

`dp[i][j]` = độ dài LCS của `a[:i]` và `b[:j]`: nếu `a[i−1] == b[j−1]` thì `dp[i−1][j−1] + 1`, ngược lại `max(dp[i−1][j], dp[i][j−1])`. Truy vết ngược để lấy chính dãy.

In [3]:
def lcs_pair(a, b):
    n, m = len(a), len(b)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(m):
            dp[i + 1][j + 1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j + 1], dp[i + 1][j])
    out, i, j = [], n, m
    while i and j:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1]); i -= 1; j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return out[::-1]

print(lcs_pair([1, 3, 4], [1, 4, 7, 9]), lcs_pair("ABCBDAB", "BDCABA"))

[1, 4] ['B', 'C', 'B', 'A']


### 3.3 Giao hai mảng đã sắp xếp bằng hai con trỏ

In [4]:
def intersect_sorted(a, b):
    i = j = 0; out = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            out.append(a[i]); i += 1; j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return out

print(intersect_sorted([2, 3, 6, 8], [1, 2, 3, 5, 6, 7, 10]))

[2, 3, 6]


### 3.4 Giao tập hợp `set.intersection` và `functools.reduce`

`set(a) & set(b)` cho phần chung (không thứ tự). `reduce(f, ds)` áp `f` lần lượt: `f(f(d0, d1), d2)…`.

In [5]:
sets = [set(a) for a in examples[1][0]]
common = reduce(lambda x, y: x & y, sets)
print(common, "->", sorted(common))

{2, 3, 6} -> [2, 3, 6]


### 3.5 Đếm tần suất với `Counter` / mảng 101 phần tử

In [6]:
cnt = [0] * 101
for arr in examples[1][0]:
    for x in arr:
        cnt[x] += 1
print("giá trị xuất hiện ở cả 3 mảng:", [v for v in range(101) if cnt[v] == 3])

giá trị xuất hiện ở cả 3 mảng: [2, 3, 6]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: các phần tử có mặt ở **cả ba** mảng là `{2, 3, 6}`. Vì mỗi mảng **tăng ngặt**, trong mỗi mảng chúng xuất hiện đúng theo thứ tự `2 < 3 < 6` ⇒ `[2, 3, 6]` là dãy con chung. Không thể dài hơn: một dãy con chung chỉ gồm phần tử có mặt ở **mọi** mảng.

Ngược lại, dãy con chung **bất kỳ** của các mảng tăng ngặt cũng phải tăng ngặt (là dãy con của một mảng tăng ngặt) ⇒ nó là một tập con của giao, viết theo thứ tự tăng.

⇒ **LCS = giao, sắp xếp tăng.** Bài toán "khó" (LCS nhiều dãy là NP-khó trong trường hợp tổng quát!) trở nên dễ nhờ giả thiết sắp xếp.

## 5. Cách 1 — Brute Force / baseline: DP LCS cổ điển gộp dần

**Ý tưởng:** `res = arrays[0]`, rồi `res = LCS(res, arrays[i])` cho từng mảng. Với dãy **tổng quát** cách gộp dần này **không** bảo đảm tối ưu (LCS của cặp đầu có thể chọn "sai" dãy), nhưng với mảng tăng ngặt thì LCS của cặp là duy nhất (= giao) nên đúng.

In [7]:
def lcs_arrays_dp(arrays):
    res = arrays[0]
    for arr in arrays[1:]:
        res = lcs_pair(res, arr)
    return res

for arrs, want in examples:
    print(lcs_arrays_dp(arrs), "| kỳ vọng", want)

[1, 4] | kỳ vọng [1, 4]
[2, 3, 6] | kỳ vọng [2, 3, 6]
[] | kỳ vọng []


**Độ phức tạp:** mỗi lần `O(L²)` với `L ≤ 100` ⇒ `O(k · L²) = 100 · 10⁴ = 10⁶` — chạy được, nhưng không tận dụng tính sắp xếp. Nút thắt: DP xét mọi cặp `(i, j)`; với dữ liệu đã sắp xếp, hai con trỏ chỉ cần `O(L)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** với các mảng tăng ngặt, `LCS(arrays) = sorted(∩ set(arrays[i]))`.

**Cách tính giao:**
- *Hai con trỏ* gộp dần: `res = intersect_sorted(res, arr)` — kết quả luôn tăng ngặt nên tiếp tục gộp được.
- *Đếm:* mỗi giá trị xuất hiện **tối đa một lần** mỗi mảng (tăng ngặt) ⇒ giá trị thuộc giao ⇔ đếm được đúng `k` lần (`k = len(arrays)`).

**Quy tắc đếm:** `cnt[v] == k` ⇔ `v` có mặt ở mọi mảng; duyệt `v` tăng dần để ra thứ tự đúng.

**Invariant (gộp dần):** sau khi gộp `i` mảng đầu, `res` = giao của chúng, tăng ngặt.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai con trỏ gộp dần (`O(Σ len)`)

In [8]:
def lcs_two_pointers_demo(arrays):
    res = arrays[0]
    print("bắt đầu:", res)
    for arr in arrays[1:]:
        res = intersect_sorted(res, arr)
        print("giao với", arr, "->", res)
    return res

print("=>", lcs_two_pointers_demo(examples[1][0]))

bắt đầu: [2, 3, 6, 8]
giao với [1, 2, 3, 5, 6, 7, 10] -> [2, 3, 6]
giao với [2, 3, 4, 6, 9] -> [2, 3, 6]
=> [2, 3, 6]


In [9]:
class SolutionTwoPointers:
    def longestCommonSubsequence(self, arrays: List[List[int]]) -> List[int]:
        res = arrays[0]
        for arr in arrays[1:]:
            res = intersect_sorted(res, arr)
            if not res:
                break                          # giao rỗng: dừng sớm
        return res

for arrs, want in examples:
    print(SolutionTwoPointers().longestCommonSubsequence(arrs) == want)

True
True
True


**Độ phức tạp:** mỗi lần giao `O(len(res) + len(arr))` ⇒ tổng `O(Σ len)`; bộ nhớ `O(L)`.

### 7.2 Cách 3 — Giao tập hợp rồi sắp xếp

In [10]:
class SolutionSets:
    def longestCommonSubsequence(self, arrays: List[List[int]]) -> List[int]:
        common = set(arrays[0])
        for arr in arrays[1:]:
            common &= set(arr)
        return sorted(common)

for arrs, want in examples:
    print(SolutionSets().longestCommonSubsequence(arrs) == want)

True
True
True


**Độ phức tạp:** `O(Σ len)` để dựng tập + `O(L log L)` sắp xếp. Ngắn gọn, nhưng **phải nhớ `sorted`** (Lỗi 1).

### 7.3 Cách 4 — Đếm tần suất trên miền `1..100` (`O(Σ len + 100)`) — lời giải nộp

In [11]:
class SolutionCount:
    def longestCommonSubsequence(self, arrays: List[List[int]]) -> List[int]:
        cnt = [0] * 101
        for arr in arrays:
            for x in arr:
                cnt[x] += 1
        k = len(arrays)
        return [v for v in range(1, 101) if cnt[v] == k]

for arrs, want in examples:
    print(SolutionCount().longestCommonSubsequence(arrs) == want)

True
True
True


Cách đếm **dựa trên** giả thiết tăng ngặt (mỗi giá trị tối đa một lần mỗi mảng). Nếu mảng chỉ **không giảm** (có lặp), phải đếm "số mảng chứa `v`" thay vì "số lần xuất hiện" (Lỗi 3).

### 7.4 Cách 5 — Duyệt mảng ngắn nhất + tìm kiếm nhị phân ở các mảng còn lại

Đáp án là tập con của mảng **ngắn nhất**. Với mỗi phần tử `v` của mảng ngắn nhất, kiểm `v` có trong mọi mảng khác bằng `bisect_left` (mảng đã sắp xếp). Tốt khi một mảng rất ngắn còn các mảng khác rất dài (và giá trị lớn, không đếm được).

In [12]:
from bisect import bisect_left

def contains(arr, v):
    i = bisect_left(arr, v)
    return i < len(arr) and arr[i] == v

class SolutionBinary:
    def longestCommonSubsequence(self, arrays: List[List[int]]) -> List[int]:
        shortest = min(arrays, key=len)
        return [v for v in shortest if all(contains(arr, v) for arr in arrays)]

for arrs, want in examples:
    print(SolutionBinary().longestCommonSubsequence(arrs) == want)

True
True
True


**Độ phức tạp:** `O(L_min · k · log L)`; không phụ thuộc miền giá trị. Thứ tự kết quả tự đúng vì duyệt theo mảng ngắn nhất (đã tăng ngặt).

## 8. TRACE từng bước

Trace hai con trỏ giao `[2,3,6,8]` và `[1,2,3,5,6,7,10]`:

| `i` | `a[i]` | `j` | `b[j]` | so sánh | hành động | kết quả |
|---|---|---|---|---|---|---|
| 0 | 2 | 0 | 1 | `2 > 1` | `j++` | |
| 0 | 2 | 1 | 2 | bằng | lấy 2 | `[2]` |
| 1 | 3 | 2 | 3 | bằng | lấy 3 | `[2,3]` |
| 2 | 6 | 3 | 5 | `6 > 5` | `j++` | |
| 2 | 6 | 4 | 6 | bằng | lấy 6 | `[2,3,6]` |
| 3 | 8 | 5 | 7 | `8 > 7` | `j++` | |
| 3 | 8 | 6 | 10 | `8 < 10` | `i++` | hết `a` |

In [13]:
def trace_intersect(a, b):
    i = j = 0; out = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            out.append(a[i]); print(f"i={i}, j={j}: {a[i]} == {b[j]} -> lấy, out={out}"); i += 1; j += 1
        elif a[i] < b[j]:
            print(f"i={i}, j={j}: {a[i]} < {b[j]} -> i++"); i += 1
        else:
            print(f"i={i}, j={j}: {a[i]} > {b[j]} -> j++"); j += 1
    print("kết quả:", out)
    return out

trace_intersect([2, 3, 6, 8], [1, 2, 3, 5, 6, 7, 10])

i=0, j=0: 2 > 1 -> j++
i=0, j=1: 2 == 2 -> lấy, out=[2]
i=1, j=2: 3 == 3 -> lấy, out=[2, 3]
i=2, j=3: 6 > 5 -> j++
i=2, j=4: 6 == 6 -> lấy, out=[2, 3, 6]
i=3, j=5: 8 > 7 -> j++
i=3, j=6: 8 < 10 -> i++
kết quả: [2, 3, 6]


[2, 3, 6]

Trace đếm trên Ví dụ 1 (chỉ in các giá trị có `cnt > 0`):

In [14]:
cnt = Counter(x for arr in examples[0][0] for x in arr)
print({v: cnt[v] for v in sorted(cnt)}, "-> chọn cnt == 2:", [v for v in sorted(cnt) if cnt[v] == 2])

{1: 2, 3: 1, 4: 2, 7: 1, 9: 1} -> chọn cnt == 2: [1, 4]


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def longestCommonSubsequence(self, arrays: List[List[int]]) -> List[int]:
        cnt = [0] * 101                       # giá trị trong 1..100
        for arr in arrays:
            for x in arr:                     # mỗi mảng tăng ngặt: x xuất hiện tối đa 1 lần
                cnt[x] += 1
        k = len(arrays)
        return [v for v in range(1, 101) if cnt[v] == k]   # duyệt tăng dần = đúng thứ tự

for arrs, want in examples:
    assert Solution().longestCommonSubsequence(arrs) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `G = ∩ᵢ set(arrays[i])` và `g = sorted(G)`.

**`g` là dãy con chung.** Với mỗi mảng `A` (tăng ngặt), các phần tử của `G` đều có trong `A` và `A` liệt kê chúng theo thứ tự tăng ⇒ `g` là dãy con của `A`.

**Mọi dãy con chung `S` có `|S| ≤ |g|`.** Mỗi phần tử của `S` có mặt trong mọi mảng ⇒ thuộc `G`. `S` là dãy con của một mảng tăng ngặt nên các phần tử của `S` **khác nhau** ⇒ `|S| ≤ |G| = |g|`.

⇒ `g` là một LCS (và thực ra là **duy nhất**: `S` có độ dài `|G|` thì là `G` theo thứ tự tăng).

**Đếm đúng:** mỗi mảng tăng ngặt chứa `v` tối đa một lần ⇒ `cnt[v]` = số mảng chứa `v` ⇒ `cnt[v] = k ⇔ v ∈ G`.

## 11. Phân tích Time / Space

Gọi `k ≤ 100` mảng, mỗi mảng `≤ L = 100` phần tử, giá trị `≤ V = 100`.

| Cách | Time | Space |
|---|---|---|
| 1. DP LCS gộp dần | `O(k · L²)` | `O(L²)` |
| 2. Hai con trỏ | `O(k · L)` | `O(L)` |
| 3. Giao tập | `O(k · L + L log L)` | `O(L)` |
| 4. Đếm | `O(k · L + V)` | `O(V)` |
| 5. Ngắn nhất + bisect | `O(L_min · k · log L)` | `O(1)` phụ |

In [16]:
rng = random.Random(1940)
def gen(k, rng):
    base = set(rng.sample(range(1, 101), 30))                 # 30 giá trị chung
    rest = [v for v in range(1, 101) if v not in base]
    return [sorted(base | set(rng.sample(rest, rng.randint(0, 70)))) for _ in range(k)]

arrs = gen(100, rng)
res = []
for name, sol in [("DP", None), ("Hai con trỏ", SolutionTwoPointers()), ("Tập", SolutionSets()), ("Đếm/Nộp", Solution())]:
    t = time.perf_counter()
    r = lcs_arrays_dp(arrs) if sol is None else sol.longestCommonSubsequence(arrs)
    res.append(tuple(r))
    print(f"{name}: độ dài {len(r)} trong {1000*(time.perf_counter()-t):.1f} ms")
assert len(set(res)) == 1

DP: độ dài 30 trong 13.8 ms
Hai con trỏ: độ dài 30 trong 0.5 ms
Tập: độ dài 30 trong 0.1 ms
Đếm/Nộp: độ dài 30 trong 0.1 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu vét cạn)

In [17]:
def lcs_brute(arrays):
    a0 = arrays[0]
    for size in range(len(a0), -1, -1):
        for idx in combinations(range(len(a0)), size):
            cand = [a0[i] for i in idx]
            if all(is_subsequence(cand, arr) for arr in arrays[1:]):
                return cand
    return []

sols = [Solution(), SolutionTwoPointers(), SolutionSets(), SolutionCount(), SolutionBinary()]
edge = examples + [
    ([[1], [1]], [1]),
    ([[1], [2]], []),
    ([[1, 2, 3], [1, 2, 3], [1, 2, 3]], [1, 2, 3]),
    ([[100], [1, 100]], [100]),
    ([[1, 5, 9], [5], [2, 5, 7]], [5]),
]
for arrs, want in edge:
    assert lcs_brute(arrs) == want
    for s in sols:
        assert s.longestCommonSubsequence(arrs) == want, (arrs, s)
    assert lcs_arrays_dp(arrs) == want
    print("OK", arrs, "->", want)

OK [[1, 3, 4], [1, 4, 7, 9]] -> [1, 4]
OK [[2, 3, 6, 8], [1, 2, 3, 5, 6, 7, 10], [2, 3, 4, 6, 9]] -> [2, 3, 6]
OK [[1, 2, 3, 4, 5], [6, 7, 8]] -> []
OK [[1], [1]] -> [1]
OK [[1], [2]] -> []
OK [[1, 2, 3], [1, 2, 3], [1, 2, 3]] -> [1, 2, 3]
OK [[100], [1, 100]] -> [100]
OK [[1, 5, 9], [5], [2, 5, 7]] -> [5]


### 12.2 Ngẫu nhiên có seed: so với vét cạn (mảng ngắn)

In [18]:
rng = random.Random(42)
for _ in range(500):
    k = rng.randint(2, 4)
    arrs = [sorted(rng.sample(range(1, 15), rng.randint(1, 8))) for _ in range(k)]
    want = lcs_brute(arrs)
    for s in sols:
        assert s.longestCommonSubsequence(arrs) == want, arrs
    assert lcs_arrays_dp(arrs) == want
print("500 bộ ngẫu nhiên: khớp vét cạn")

500 bộ ngẫu nhiên: khớp vét cạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Giao tập hợp nhưng quên sắp xếp

In [19]:
def wrong_unsorted(arrays):
    common = set(arrays[0])
    for arr in arrays[1:]:
        common &= set(arr)
    return list(common)                                  # SAI: thứ tự của set không bảo đảm

arrs = [[3, 40, 72], [3, 40, 72, 99]]
print("list(set):", wrong_unsorted(arrs), "| đúng:", Solution().longestCommonSubsequence(arrs))
print("với số lớn hơn thứ tự lộn xộn hơn:", list({1000, 8, 64}))

list(set): [40, 3, 72] | đúng: [3, 40, 72]
với số lớn hơn thứ tự lộn xộn hơn: [1000, 8, 64]


**Vì sao sai:** `set` **không có thứ tự** theo đặc tả; đôi khi kết quả trông như tăng dần (với số nhỏ, liền nhau) nhưng ở ví dụ trên đã ra `[40, 3, 72]` — không được dựa vào. Kết quả phải là **dãy con**, tức theo đúng thứ tự tăng. **Sửa:** `sorted(common)`.

### Lỗi 2 — Hai con trỏ tăng cả hai khi khác nhau

In [20]:
def wrong_two_pointer(a, b):
    i = j = 0; out = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            out.append(a[i])
        i += 1; j += 1                                   # SAI: phải chỉ tăng con trỏ nhỏ hơn
    return out

print("sai:", wrong_two_pointer([2, 3, 6, 8], [1, 2, 3, 5, 6, 7, 10]), "| đúng:", intersect_sorted([2, 3, 6, 8], [1, 2, 3, 5, 6, 7, 10]))

sai: [] | đúng: [2, 3, 6]


**Vì sao sai:** khi `a[i] ≠ b[j]`, chỉ phần tử **nhỏ hơn** chắc chắn không khớp với phần còn lại của mảng kia; tăng cả hai sẽ bỏ lỡ cặp khớp lệch vị trí.

### Lỗi 3 — Đếm số lần xuất hiện khi mảng có thể lặp (nếu đề đổi thành "không giảm")

In [21]:
def count_occurrences(arrays):
    cnt = Counter(x for arr in arrays for x in arr)
    return [v for v in sorted(cnt) if cnt[v] >= len(arrays)]   # SAI nếu có lặp

arrs = [[1, 1, 2], [2, 3]]
print("sai:", count_occurrences(arrs), "| đúng (đếm số mảng chứa v):",
      sorted(set(arrs[0]) & set(arrs[1])))

sai: [1, 2] | đúng (đếm số mảng chứa v): [2]


**Vì sao sai (với biến thể có lặp):** `1` xuất hiện hai lần trong mảng đầu nên `cnt[1] = 2 = k` dù mảng thứ hai không có `1`. Với đề gốc (tăng ngặt) cách đếm an toàn; cần nhận ra **giả thiết** mình đang dựa vào. (Và với mảng có lặp, LCS còn phải tính bội số: `min` số lần xuất hiện.)

### Lỗi 4 — Áp "LCS = giao" cho mảng **không** sắp xếp

In [22]:
def lcs_general_brute(arrays):
    return lcs_brute(arrays)

arrs = [[1, 2, 3], [3, 2, 1]]
print("giao:", sorted(set(arrs[0]) & set(arrs[1])), "| LCS thật:", lcs_general_brute(arrs))

giao: [1, 2, 3] | LCS thật: [1]


**Vì sao sai:** khi mảng không sắp xếp, giao `{1,2,3}` theo thứ tự nào cũng **không** là dãy con của cả hai; LCS thật chỉ dài 1. Lời giải của bài **chỉ đúng** nhờ giả thiết tăng ngặt.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Thấy giả thiết "đã sắp xếp / tăng ngặt"** ⇒ tự hỏi nó biến bài khó thành bài nào dễ hơn (ở đây LCS → giao).
- Giao nhiều mảng sắp xếp ⇒ hai con trỏ gộp dần, hoặc đếm nếu giá trị nhỏ.
- `set` không có thứ tự — luôn `sorted` khi thứ tự quan trọng.
- Miền giá trị nhỏ (≤ 100, ≤ 10⁵) ⇒ mảng đếm thay cho băm.

## 15. Biến thể của bài

1. **LeetCode 349 / 350 — Intersection of Two Arrays (I/II):** giao không lặp / có lặp.
2. **LeetCode 1143 — LCS của hai chuỗi** (tổng quát, DP `O(nm)`).
3. **LCS của ba chuỗi tổng quát** ⇒ DP 3 chiều `O(n·m·p)` (Bài tập 5).
4. **Mảng không giảm (có lặp)** ⇒ giao đa tập: `min` số lần xuất hiện.
5. **LeetCode 1213 — Intersection of Three Sorted Arrays:** đúng bài này với `k = 3`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP LCS gộp dần | LCS từng cặp | `O(kL²)` | `O(L²)` | kiểm chứng; dãy tổng quát 2 dãy |
| 2. Hai con trỏ | giao mảng sắp xếp | `O(kL)` | `O(L)` | giá trị lớn, mảng sắp xếp |
| 3. Giao tập | `set` + `sorted` | `O(kL + L log L)` | `O(L)` | code ngắn |
| 4. Đếm | `cnt[v] == k` | `O(kL + V)` | `O(V)` | **nộp bài** (V nhỏ) |
| 5. Ngắn nhất + bisect | kiểm từng `v` bằng nhị phân | `O(L_min·k·log L)` | `O(1)` phụ | một mảng rất ngắn, giá trị lớn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `[[1,2],[2,3]]`, `[[5],[5],[5]]`, `[[1,3],[2,4]]`?

**Bài 2 (tính tay):** Trace hai con trỏ gộp dần cho Ví dụ 2.

**Bài 3 (điền TODO):** Viết `intersect_k_pointers(arrays)` — dùng **k con trỏ đồng thời** (mỗi mảng một con trỏ), mỗi bước tăng mọi con trỏ đang trỏ vào giá trị nhỏ hơn giá trị lớn nhất hiện tại.

**Bài 4 (tự viết lại):** Viết `common_multiset(arrays)` cho biến thể mảng **không giảm**: trả về giao đa tập (theo thứ tự tăng, mỗi giá trị lặp `min` số lần).

**Bài 5 (mini challenge):** Viết `lcs3(a, b, c)` — LCS của **ba** dãy tổng quát (không sắp xếp), DP 3 chiều, đối chiếu vét cạn.

In [23]:
def intersect_k_pointers(arrays):
    # TODO
    pass

def common_multiset(arrays):
    # TODO
    pass

def lcs3(a, b, c):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Giao | `[2]`, `[5]` | `[]` |
| 2 | Giao mảng 1, 2 trước | `[2,3,6]` | Giao với mảng 3 vẫn `[2,3,6]` |
| 3 | `ptr = [0]*k` | `hi = max(arr[ptr])`; nếu mọi giá trị bằng `hi` thì lấy và tăng hết | Ngược lại tăng các con trỏ có giá trị `< hi` |
| 4 | `Counter` từng mảng | Giao `Counter` bằng `&` (lấy min) | `sorted(c.elements())` |
| 5 | `dp[i][j][l]` | Bằng nhau cả ba thì `+1` | Ngược lại max của 3 cách bỏ một phần tử |

## 19. Đáp án tham khảo

In [24]:
r1 = [Solution().longestCommonSubsequence(a) for a in ([[1, 2], [2, 3]], [[5], [5], [5]], [[1, 3], [2, 4]])]
print("Bài 1:", r1); assert r1 == [[2], [5], []]
print("Bài 2:"); assert lcs_two_pointers_demo(examples[1][0]) == [2, 3, 6]

Bài 1: [[2], [5], []]
Bài 2:
bắt đầu: [2, 3, 6, 8]
giao với [1, 2, 3, 5, 6, 7, 10] -> [2, 3, 6]
giao với [2, 3, 4, 6, 9] -> [2, 3, 6]


In [25]:
def intersect_k_pointers(arrays):
    k = len(arrays); ptr = [0] * k; out = []
    while all(ptr[i] < len(arrays[i]) for i in range(k)):
        vals = [arrays[i][ptr[i]] for i in range(k)]
        hi = max(vals)
        if all(v == hi for v in vals):
            out.append(hi); ptr = [p + 1 for p in ptr]
        else:
            for i in range(k):
                if vals[i] < hi:
                    ptr[i] += 1
    return out

def common_multiset(arrays):
    c = Counter(arrays[0])
    for arr in arrays[1:]:
        c &= Counter(arr)
    return sorted(c.elements())

rng = random.Random(3)
for _ in range(400):
    k = rng.randint(2, 5)
    arrs = [sorted(rng.sample(range(1, 30), rng.randint(1, 12))) for _ in range(k)]
    assert intersect_k_pointers(arrs) == Solution().longestCommonSubsequence(arrs)
    dup = [sorted(rng.choice(range(1, 6)) for _ in range(rng.randint(1, 8))) for _ in range(k)]
    assert common_multiset(dup) == lcs_brute(dup)            # mảng không giảm: LCS = giao đa tập
print("Bài 3, 4: OK", intersect_k_pointers(examples[1][0]), common_multiset([[1, 1, 2, 2], [1, 2, 2, 3]]))

Bài 3, 4: OK [2, 3, 6] [1, 2, 2]


In [26]:
def lcs3(a, b, c):
    n, m, p = len(a), len(b), len(c)
    dp = [[[0] * (p + 1) for _ in range(m + 1)] for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            for l in range(1, p + 1):
                if a[i - 1] == b[j - 1] == c[l - 1]:
                    dp[i][j][l] = dp[i - 1][j - 1][l - 1] + 1
                else:
                    dp[i][j][l] = max(dp[i - 1][j][l], dp[i][j - 1][l], dp[i][j][l - 1])
    return dp[n][m][p]

rng = random.Random(9)
for _ in range(200):
    a, b, c = ([rng.randint(1, 4) for _ in range(rng.randint(1, 7))] for _ in range(3))
    assert lcs3(a, b, c) == len(lcs_brute([a, b, c]))
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK", lcs3([1, 2, 3], [3, 2, 1], [2, 3, 1]))

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cnt = [0] * 101
for arr in arrays:
    for x in arr:
        cnt[x] += 1                      # tăng ngặt: mỗi mảng góp tối đa 1
return [v for v in range(1, 101) if cnt[v] == len(arrays)]
```

**Checklist:**
- [ ] Dựa vào giả thiết **tăng ngặt** ⇒ LCS = giao theo thứ tự tăng?
- [ ] Kết quả **sắp xếp tăng** (không trả `list(set)`)?
- [ ] Hai con trỏ: chỉ tăng con trỏ trỏ vào giá trị **nhỏ hơn**?
- [ ] Mảng đếm đủ chỗ cho giá trị `100` (kích thước 101)?
- [ ] Giao rỗng ⇒ trả `[]`.